In [1]:
import pandas as pd
import numpy as np
from model import initialize_model
import torch
import os

data_dir = '/Users/marcomaluf/Desktop/Unfinished Projects/paper/preprocessed_data'
save_dir = '/Users/marcomaluf/Desktop/Unfinished Projects/paper/final_data'

data = os.listdir(data_dir)
model = initialize_model()

/Users/marcomaluf/Desktop/Unfinished Projects/paper/methodology/model.py:46: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(MODEL_PATH, map_location=t

In [2]:
for csv in data:
    df = pd.read_csv(f'{data_dir}/{csv}').drop(columns=['Unnamed: 0'])

    segments = [df.iloc[i:i+32, :13] for i in range(0,len(df),64)]
    all_xG = []
    for seg in segments:
        all_xG.append(model.predict_xG(torch.from_numpy(seg.to_numpy().astype(np.float32)).unsqueeze(0)))

    df.loc[:, 'xG'] = np.nan
    for i, xG in zip(range(0,len(df),64), all_xG):
        df.loc[i:i+64-1, 'xG'] = np.round(float(xG), 3)

    df.to_csv(f'{save_dir}/{csv[:-8]}_final.csv')

In [3]:
lets_see = pd.read_csv('/Users/marcomaluf/Desktop/Unfinished Projects/paper/final_data/OCE P-E bruv vs BOW G3 2026-01-18.18.16_final.csv').drop(columns=['Unnamed: 0'])

In [4]:
lets_see

,pos_x,pos_y,pos_z,vel_x,vel_y,vel_z,dist_to_goal,angle_to_goal,players_btwn_ball_and_goal,players_within_400_of_ball,ball_state,team1_cluster,team2_cluster,label,base_df_index,xG
0,2.387569,0.716586,2.375487,-0.027947,-0.087968,-0.080042,0.825781,0.062224,0.0,0.000000,0.181842,-0.377696,0.317082,0,760,0.001
1,2.374532,0.714807,2.393819,-0.027947,-0.087968,-0.080042,0.823774,0.062966,0.0,0.000000,0.182778,-0.377004,0.316078,0,761,0.001
2,2.355005,0.712141,2.418590,-0.027947,-0.087968,-0.080042,0.820725,0.064087,0.0,0.000000,0.182054,-0.376686,0.316001,0,762,0.001
3,2.345250,0.710809,2.429751,-0.027947,-0.087968,-0.080042,0.819182,0.064652,0.0,0.000000,0.181067,-0.377027,0.316671,0,763,0.001
4,2.332259,0.709032,2.443381,-0.027947,-0.087968,-0.080042,0.817106,0.065409,0.0,0.000000,0.180824,-0.376765,0.316490,0,764,0.001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1019,0.279227,-1.220133,-0.154342,-1.052531,-0.539204,-0.072607,0.107727,0.616354,0.0,0.333333,-0.032731,-0.380481,0.391391,0,9677,0.936
1020,0.304898,-1.233184,-0.132929,-1.052007,-0.538977,-0.091945,0.103177,0.634923,0.0,0.333333,-0.033854,-0.379507,0.390791,0,9678,0.936
1021,0.351288,-1.244878,-0.093362,-1.050696,-0.538408,-0.140292,0.103560,0.646466,0.0,0.333333,-0.034863,-0.378888,0.390509,0,9679,0.936
1022,0.408937,-1.259494,-0.046644,-1.049647,-0.537953,-0.178897,0.106842,0.657379,0.0,0.333333,-0.038603,-0.378516,0.391384,0,9680,0.936


In [7]:
total_goals = lets_see[lets_see['xG'] != -1].label.sum()/64
total_xG = lets_see[lets_see['xG'] != -1].xG.sum()/64
print(f'Total Goals: {total_goals} | Total xG: {total_xG}')

Total Goals: 3.0 | Total xG: 6.596
